In [1]:
!pip install pandas numpy scikit-learn nltk

Import Libraries

In [2]:
import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

from sklearn.model_selection import train_test_split

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [3]:
df = pd.read_csv("sentiment_analysis.csv")
print(df.head())

                                                text sentiment
0  I experienced this emotion when my grandfather...   sadness
1   when I first moved in , I walked everywhere ....   neutral
2  ` Oh ! " she bleated , her voice high and rath...     anger
3  However , does the right hon. Gentleman recogn...      fear
4  My boyfriend didn't turn up after promising th...   sadness


Data Understanding

In [4]:
print("\nDataset Shape:")
print(df.shape)

print("\nColumn Names:")
print(df.columns)

print("\nClass Distribution:")
print(df["sentiment"].value_counts())

print("\nSample Text:")
print(df["text"].iloc[0])


Dataset Shape:
(11327, 2)

Column Names:
Index(['text', 'sentiment'], dtype='object')

Class Distribution:
sentiment
joy        2326
sadness    2317
anger      2259
neutral    2254
fear       2171
Name: count, dtype: int64

Sample Text:
I experienced this emotion when my grandfather passed away.


NLP Preprocessing

In [5]:
stop_words = set(stopwords.words("english"))

stemmer = PorterStemmer()

def preprocess_text(text):

    # convert to lowercase
    text = text.lower()

    # remove urls
    text = re.sub(r"http\S+|www\S+", "", text)

    # remove punctuation & special characters
    text = re.sub(r"[^a-z\s]", "", text)

    # tokenization
    words = text.split()

    # remove stopwords
    words = [w for w in words if w not in stop_words]

    # stemming
    words = [stemmer.stem(w) for w in words]

    # join words
    clean_text = " ".join(words)

    return clean_text


# apply preprocessing
df["clean_text"] = df["text"].apply(preprocess_text)

print(df[["text","clean_text"]].head())

                                                text  \
0  I experienced this emotion when my grandfather...   
1   when I first moved in , I walked everywhere ....   
2  ` Oh ! " she bleated , her voice high and rath...   
3  However , does the right hon. Gentleman recogn...   
4  My boyfriend didn't turn up after promising th...   

                                          clean_text  
0                 experienc emot grandfath pass away  
1  first move walk everywher within week purs sto...  
2                   oh bleat voic high rather indign  
3  howev right hon gentleman recognis profound di...  
4                   boyfriend didnt turn promis come  


Train Test Split

In [6]:
X = df["clean_text"]

y = df["sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))

Train size: 9061
Test size: 2266


Feature Engineering

1️ Bag of Words

In [7]:
bow = CountVectorizer()

X_train_bow = bow.fit_transform(X_train)

X_test_bow = bow.transform(X_test)

print("BoW Shape:", X_train_bow.shape)

BoW Shape: (9061, 8112)


2 . TF-IDF

In [8]:
tfidf = TfidfVectorizer()

X_train_tfidf = tfidf.fit_transform(X_train)

X_test_tfidf = tfidf.transform(X_test)

print("TFIDF Shape:", X_train_tfidf.shape)

TFIDF Shape: (9061, 8112)


Model Training

In [9]:
lr_model = LogisticRegression(max_iter=200)

lr_model.fit(X_train_tfidf, y_train)

lr_pred = lr_model.predict(X_test_tfidf)

or
Naive Bayes

In [10]:
nb_model = MultinomialNB()

nb_model.fit(X_train_tfidf, y_train)

nb_pred = nb_model.predict(X_test_tfidf)

or   Decision Tree

In [11]:
dt_model = DecisionTreeClassifier()

dt_model.fit(X_train_bow, y_train)

dt_pred = dt_model.predict(X_test_bow)

Model Evaluation

In [13]:
def evaluate_model(y_test, pred, name):

    print("\n====================")

    print(name)

    acc = accuracy_score(y_test, pred)

    precision = precision_score(y_test, pred, average="weighted")

    recall = recall_score(y_test, pred, average="weighted")

    f1 = f1_score(y_test, pred, average="weighted")

    print("Accuracy:", acc)

    print("Precision:", precision)

    print("Recall:", recall)

    print("F1 Score:", f1)

    print("\nClassification Report:\n")

    print(classification_report(y_test, pred))

    return acc, precision, recall, f1

In [14]:
lr_results = evaluate_model(y_test, lr_pred, "Logistic Regression")

nb_results = evaluate_model(y_test, nb_pred, "Naive Bayes")

dt_results = evaluate_model(y_test, dt_pred, "Decision Tree")


Logistic Regression
Accuracy: 0.6910856134157105
Precision: 0.6934728567930074
Recall: 0.6910856134157105
F1 Score: 0.6909401183516524

Classification Report:

              precision    recall  f1-score   support

       anger       0.69      0.70      0.69       464
        fear       0.70      0.67      0.69       435
         joy       0.68      0.63      0.65       449
     neutral       0.65      0.76      0.70       449
     sadness       0.75      0.69      0.72       469

    accuracy                           0.69      2266
   macro avg       0.69      0.69      0.69      2266
weighted avg       0.69      0.69      0.69      2266


Naive Bayes
Accuracy: 0.6266548984995587
Precision: 0.6378570521479177
Recall: 0.6266548984995587
F1 Score: 0.6245491067334343

Classification Report:

              precision    recall  f1-score   support

       anger       0.65      0.67      0.66       464
        fear       0.71      0.63      0.67       435
         joy       0.52      0.66 

Model Comparison

In [15]:
results_df = pd.DataFrame({

    "Model":[
        "Logistic Regression",
        "Naive Bayes",
        "Decision Tree"
    ],

    "Accuracy":[
        lr_results[0],
        nb_results[0],
        dt_results[0]
    ],

    "Precision":[
        lr_results[1],
        nb_results[1],
        dt_results[1]
    ],

    "Recall":[
        lr_results[2],
        nb_results[2],
        dt_results[2]
    ],

    "F1 Score":[
        lr_results[3],
        nb_results[3],
        dt_results[3]
    ]

})

print("\nFinal Comparison Table:\n")

print(results_df)


Final Comparison Table:

                 Model  Accuracy  Precision    Recall  F1 Score
0  Logistic Regression  0.691086   0.693473  0.691086  0.690940
1          Naive Bayes  0.626655   0.637857  0.626655  0.624549
2        Decision Tree  0.604148   0.617778  0.604148  0.601320


Final Insights:


The complete NLP pipeline successfully transformed raw text into meaningful features and enabled effective sentiment prediction.

Key findings:

Proper preprocessing improves model accuracy

*   TF-IDF provides better feature representation than Bag of Words
*   Logistic Regression achieved the best balance between bias and variance
*   complex than binary sentiment datasets, which slightly lowers accuracy
*   List item
*   Traditional ML algorithms can still perform well for NLP classification tasks
*   Multi-class emotion datasets (joy, anger, sadness, fear, neutral) are more

